In [ ]:
%pip install pandas numpy

# 🚀 TP Synthèse : Data Wrangling & Sauvetage Spatial

**Mission :** Le vaisseau spatial de croisière *Spaceship Titanic* a percuté une anomalie spatio-temporelle. Près de la moitié des passagers ont été téléportés dans une dimension alternative ! La base de données du vaisseau a été gravement corrompue lors du choc. 

En tant que Data Engineer d'urgence, vous devez nettoyer et préparer les données (Wrangling) pour que notre IA de sauvetage puisse ensuite prédire qui a disparu et qui est encore à bord.

*Dataset source : [Kaggle - Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/data)*

In [ ]:
# Importation de notre boîte à outils
import pandas as pd
import numpy as np

df = pd.read_csv('train.csv')

print("Aperçu des données corrompues :")
display(df.head(3))

## 🛠️ Étape 1 : Gestion des valeurs manquantes (`_21_valeurs_manquantes`)

L'anomalie a effacé certaines informations cruciales. Une IA ne peut pas s'entraîner sur du vide (`NaN`). Nous devons boucher ces trous de manière intelligente.

1. **Analyse :** Regardons l'étendue des dégâts.
2. **Imputation :** Remplaçons l'âge manquant par l'âge médian du vaisseau, et le statut VIP par la valeur la plus fréquente (le mode).

In [ ]:
# 1. Bilan des valeurs manquantes
print("--- Trous dans la base de données ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

# 2. Remplissage (Imputation)
# On remplace les âges manquants par la médiane (moins sensible aux valeurs extrêmes que la moyenne)
df['Age'] = df['Age'].fillna(df['Age'].median())

# On remplace le statut VIP manquant par 'False' (statistiquement le plus probable)
df['VIP'] = df['VIP'].fillna(False)

# 3. Suppression
# Pour la cabine, l'information est trop complexe pour l'inventer. On supprime les lignes sans cabine.
df = df.dropna(subset=['Cabin'])

print(f"\nNettoyage effectué. Il reste {len(df)} passagers viables pour l'analyse.")

## 🪚 Étape 2 : Harmonisation et Parsing (`_22_harmonisation`)

La colonne `Cabin` est un cauchemar : elle contient le pont, le numéro et le côté du vaisseau, tout collé ensemble (ex: `B/0/P`). L'IA ne comprendra rien à cette chaîne de caractères. 
Nous devons la parser (la découper) en trois colonnes distinctes et harmonisées.

In [ ]:
# A VOUS DE JOUER :
# Utilisez la méthode .str.split() de Pandas pour découper la colonne 'Cabin'

# 1. On sépare la chaîne de caractères sur le délimiteur '/'
# expand=True permet de créer un nouveau DataFrame avec 3 colonnes
cabins_split = df['Cabin'].str.split('/', expand=True)

# 2. On assigne ces nouvelles colonnes à notre base principale
df['Deck'] = cabins_split[0]
df['Num'] = cabins_split[1]
df['Side'] = cabins_split[2]

# 3. On supprime l'ancienne colonne devenue inutile
df = df.drop(columns=['Cabin'])

display(df[['Name', 'Deck', 'Num', 'Side']].head(3))

## 🧪 Étape 3 : Transformation et Feature Engineering (`_23_transformation`)

Le *Feature Engineering* consiste à créer de nouvelles variables plus pertinentes à partir de celles existantes. 
Nous avons les dépenses des passagers éparpillées dans 5 boutiques différentes (`RoomService`, `FoodCourt`, `ShoppingMall`, `Spa`, `VRDeck`). 

Pour aider notre modèle, créons une nouvelle variable `Total_Spent` qui regroupe toutes ces dépenses !

In [ ]:
# On liste les colonnes financières
colonnes_depenses = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

# On comble les valeurs manquantes financières par 0 (s'ils n'ont rien dépensé, c'est 0)
df[colonnes_depenses] = df[colonnes_depenses].fillna(0)

# A VOUS DE JOUER :
# Créez la colonne 'Total_Spent' en additionnant les 5 colonnes
df['Total_Spent'] = df['RoomService'] + df['FoodCourt'] + df['ShoppingMall'] + df['Spa'] + df['VRDeck']

# Vérification pour le passager VIP
print("Les plus gros dépensiers du vaisseau :")
display(df[['Name', 'VIP', 'Total_Spent']].sort_values(by='Total_Spent', ascending=False).head(3))

## 🧮 Étape 4 : Encodage des catégories (`_24_encodage`)

L'algorithme de sauvetage ne comprend que les mathématiques. Or, nous avons des planètes d'origine (`HomePlanet`) sous forme de texte ("Earth", "Europa", "Mars"). 
Nous allons utiliser la technique du **One-Hot Encoding** (création de variables muettes) pour transformer ce texte en colonnes binaires composées de 0 et de 1.

In [ ]:
print(f"Format avant encodage : {df.shape}")

# 1. One-Hot Encoding avec pd.get_dummies() sur les planètes et les ponts
# drop_first=True permet d'éviter la multicolinéarité
df_encode = pd.get_dummies(df, columns=['HomePlanet', 'Destination', 'Deck', 'Side'], drop_first=True)

# 2. Conversion des booléens (True/False) en entiers (1/0)
colonnes_bool = ['CryoSleep', 'VIP', 'Transported']
for col in colonnes_bool:
    df_encode[col] = df_encode[col].astype(float) # On passe en float car il reste peut-être des NaN dans CryoSleep

# On supprime les variables textuelles inexploitables par l'IA (le nom et l'ID) pour la version finale
df_finale = df_encode.drop(columns=['PassengerId', 'Name'])

print(f"Format après encodage : {df_finale.shape}")
print("\n✅ Base de données prête pour l'algorithme d'Intelligence Artificielle !")
display(df_finale.head(3))